### Group 8

In [3]:
%pip install kagglehub
%pip install torch
%pip install torchvision
%pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 129.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import kagglehub
import numpy as np
import pandas as pd
import os


/home/xjv5nf/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Get device for training
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Download latest version
path = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")

df = pd.read_csv(path + "/2022/heart_2022_no_nans.csv")
df.head(10)


Using cpu device


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,4.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,None of them,No,...,1.60,71.67,27.99,No,No,Yes,Yes,"Yes, received Tdap",No,No
1,Alabama,Male,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,6.0,None of them,No,...,1.78,95.25,30.13,No,No,Yes,Yes,"Yes, received tetanus shot but not sure what type",No,No
2,Alabama,Male,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,"6 or more, but not all",No,...,1.85,108.86,31.66,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
3,Alabama,Female,Fair,5.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,None of them,No,...,1.70,90.72,31.32,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
4,Alabama,Female,Good,3.0,15.0,Within past year (anytime less than 12 months ...,Yes,5.0,1 to 5,No,...,1.55,79.38,33.07,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
5,Alabama,Male,Good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,None of them,No,...,1.85,120.20,34.96,Yes,Yes,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
6,Alabama,Female,Good,3.0,0.0,Within past year (anytime less than 12 months ...,Yes,8.0,"6 or more, but not all",No,...,1.63,88.00,33.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
7,Alabama,Male,Fair,5.0,0.0,Within past year (anytime less than 12 months ...,Yes,8.0,1 to 5,Yes,...,1.75,74.84,24.37,No,Yes,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
8,Alabama,Male,Good,2.0,0.0,5 or more years ago,No,6.0,None of them,No,...,1.70,78.02,26.94,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,Yes
9,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,None of them,No,...,1.68,63.50,22.60,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No


## Model 1: Neural Network for Predicting Heart Attacks

In [3]:
# Turn categorical data in true/false columns
df = pd.get_dummies(df, drop_first=True)

In [4]:
# Split the data into train, test, validation
target_name = "HadHeartAttack_Yes"
feature_names = [c for c in df.columns if c != target_name]

X = df[feature_names].to_numpy(dtype=np.float32)
y = df[target_name].to_numpy(dtype=np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.2,
                                                        stratify=y,
                                                        random_state=47)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, 
                                                    test_size=.5,
                                                    stratify=y_temp,
                                                    random_state=47)


In [5]:
# Train the scaler on the training data and apply it to train, validation, and
# test data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [6]:
# Define this problems Dataset
class HeartAttackDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features)
        self.labels = torch.tensor(labels).unsqueeze(1)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return len(self.labels)

In [7]:
# Create Datasets and Dataloaders for train/val/test data
train_dataset = HeartAttackDataset(X_train, y_train)
val_dataset = HeartAttackDataset(X_val, y_val)
test_dataset = HeartAttackDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32)
test_dataloader = DataLoader(test_dataset, batch_size=32)


In [8]:
# Define the model, loss function, and optimizer
model = nn.Sequential(
            nn.Linear(X_train.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 1)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
# Train the model and score it against the validation data
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_dataloader):
        inputs = inputs.to(device)
        targets = targets.to(device)
        # 1. Zero the gradients from the previous step
        optimizer.zero_grad()

        # 2. Forward pass: compute predicted outputs by passing inputs to the model
        outputs = model(inputs)

        # 3. Calculate the loss
        loss = criterion(outputs, targets)

        # 4. Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()

        # 5. Optimization step: update parameters based on gradients
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for inputs, targets in val_dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            # outputs are raw logits; apply sigmoid to get probabilities [0, 1]
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    train_loss = train_loss / len(train_dataloader)
    val_loss = val_loss / len(val_dataloader)
    val_acc = accuracy_score(all_targets, all_preds)
    val_recall = recall_score(all_targets, all_preds, zero_division=0)
    val_precision = precision_score(all_targets, all_preds, zero_division=0)
    val_f1 = f1_score(all_targets, all_preds, zero_division=0)

    # ==========================
    # 3. PRINT STATS
    # ==========================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Recall: {val_recall:.4f} | "
        f"F1: {val_f1:.4f}"
    )

print("Training finished!")



Epoch [1/10] | Train Loss: 0.1534 | Val Loss: 0.1463 | Val Acc: 0.9485 | Recall: 0.2487 | F1: 0.3454
Epoch [2/10] | Train Loss: 0.1440 | Val Loss: 0.1455 | Val Acc: 0.9487 | Recall: 0.2256 | F1: 0.3242
Epoch [3/10] | Train Loss: 0.1423 | Val Loss: 0.1466 | Val Acc: 0.9486 | Recall: 0.1862 | F1: 0.2833
Epoch [4/10] | Train Loss: 0.1410 | Val Loss: 0.1479 | Val Acc: 0.9477 | Recall: 0.2964 | F1: 0.3823
Epoch [5/10] | Train Loss: 0.1398 | Val Loss: 0.1476 | Val Acc: 0.9481 | Recall: 0.2375 | F1: 0.3333
Epoch [6/10] | Train Loss: 0.1388 | Val Loss: 0.1475 | Val Acc: 0.9485 | Recall: 0.2010 | F1: 0.2987
Epoch [7/10] | Train Loss: 0.1378 | Val Loss: 0.1484 | Val Acc: 0.9485 | Recall: 0.1906 | F1: 0.2876
Epoch [8/10] | Train Loss: 0.1368 | Val Loss: 0.1499 | Val Acc: 0.9480 | Recall: 0.2189 | F1: 0.3149
Epoch [9/10] | Train Loss: 0.1359 | Val Loss: 0.1486 | Val Acc: 0.9485 | Recall: 0.2234 | F1: 0.3215
Epoch [10/10] | Train Loss: 0.1348 | Val Loss: 0.1507 | Val Acc: 0.9483 | Recall: 0.1876 | 